### **1. Character**

In [5]:
import pandas as pd

prediction_csv = (
    "../02_data_results/"
    "character_i_will_marry_your_brother_episode1.csv"
)

test_df = pd.read_csv(prediction_csv)

print(test_df.columns.tolist())
print(test_df.head())

['window', 'character_id', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2']
           window  character_id   class  confidence          x1           y1  \
0  window_001.png             0  person    0.837996  170.525299   862.911377   
1  window_002.png             0  person    0.930577    0.000000  1041.903076   
2  window_003.png             0  person    0.890223   83.484680   624.140869   
3  window_003.png             1  person    0.685427  290.078766   425.287842   
4  window_005.png             0  person    0.938792    0.391531   703.131287   

           x2           y2  
0  321.712341  1154.709961  
1  688.250793  2321.263672  
2  309.543396   885.728760  
3  686.721436   962.464050  
4  677.965210  1843.748413  


In [6]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import re


# 1. PATHS

# Label Studio export
labelstudio_dir = Path(
    "../02_data_results/labelstudio_i_will_marry_your_brother_episode1"
)

gt_dir = labelstudio_dir / "labels"

# Original images
image_dir = Path(
    "../03_webtoon_raw/window_i_will_marry_your_brother_episode1"
)

# Character model predictions
prediction_csv = Path(
    "../02_data_results/character_i_will_marry_your_brother_episode1.csv"
)

# Evaluation output
output_dir = Path(
    "../02_data_results/evaluation_i_will_marry_your_brother_episode1"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# 2. SETTINGS

CHARACTER_CLASS_ID = 0
IOU_THRESHOLD = 0.50


# 3. FUNCTION: EXTRACT WINDOW NAME

def get_window_name(filename):
    """
    Examples:

    fcfc06f1-window_001.txt
        -> window_001

    window_001.png
        -> window_001
    """

    match = re.search(
        r"(window_\d+)",
        str(filename)
    )

    if match:
        return match.group(1)

    return Path(filename).stem


# 4. FUNCTION: CALCULATE IoU

def calculate_iou(box1, box2):
    """
    box format:
    [x1, y1, x2, y2]
    """

    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])

    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    # No overlap
    if (
        x_right <= x_left
        or y_bottom <= y_top
    ):
        return 0.0

    intersection = (
        (x_right - x_left)
        *
        (y_bottom - y_top)
    )

    area1 = (
        (box1[2] - box1[0])
        *
        (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0])
        *
        (box2[3] - box2[1])
    )

    union = (
        area1
        + area2
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


# 5. FIND ALL IMAGES

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
}

image_paths = [
    p
    for p in image_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

print(
    "Images found:",
    len(image_paths)
)


# Create:
# window_001 -> image path

image_map = {}

for path in image_paths:

    window = get_window_name(
        path.name
    )

    image_map[window] = path


print(
    "Unique windows:",
    len(image_map)
)


# 6. LOAD MODEL PREDICTIONS

pred_df = pd.read_csv(
    prediction_csv
)

print("\nPrediction CSV columns:")
print(pred_df.columns.tolist())


# Convert:
# window_001.png
# ->
# window_001

pred_df["window"] = (
    pred_df["window"]
    .astype(str)
    .apply(get_window_name)
)


print(
    "\nPredicted boxes:",
    len(pred_df)
)

print(
    "Windows with predictions:",
    pred_df["window"].nunique()
)

print("\nPrediction classes:")
print(pred_df["class"].value_counts())


# 7. LOAD LABEL STUDIO CHARACTER GT

gt_records = []

label_files = sorted(
    gt_dir.glob("*.txt")
)


print(
    "\nLabel files:",
    len(label_files)
)


for label_file in label_files:

    window = get_window_name(
        label_file.name
    )

    # Find corresponding image

    if window not in image_map:

        print(
            "WARNING: image not found for",
            label_file.name
        )

        continue


    image_path = image_map[window]


    # Read actual image dimensions
    with Image.open(image_path) as img:

        W, H = img.size

    # Read YOLO GT annotation

    with open(
        label_file,
        "r"
    ) as f:

        lines = f.readlines()


    for line in lines:

        parts = (
            line
            .strip()
            .split()
        )


        if len(parts) != 5:
            continue


        class_id = int(
            parts[0]
        )


        # CHARACTER ONLY
        if class_id != CHARACTER_CLASS_ID:
            continue


        x_center = float(
            parts[1]
        )

        y_center = float(
            parts[2]
        )

        width = float(
            parts[3]
        )

        height = float(
            parts[4]
        )


        # YOLO normalized xywh
        # ->
        # pixel xyxy
      
        x1 = (
            x_center
            - width / 2
        ) * W

        y1 = (
            y_center
            - height / 2
        ) * H

        x2 = (
            x_center
            + width / 2
        ) * W

        y2 = (
            y_center
            + height / 2
        ) * H


        gt_records.append(
            {
                "window": window,

                "image_width": W,
                "image_height": H,

                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            }
        )


gt_df = pd.DataFrame(
    gt_records
)


print(
    "\nGT character boxes:",
    len(gt_df)
)


# 8. CHECK WINDOWS

all_windows = sorted(
    image_map.keys()
)


print(
    "Total windows for evaluation:",
    len(all_windows)
)


# 9. MATCH GT AND PREDICTIONS

evaluation_records = []


for window in all_windows:

    # Ground truth
    if len(gt_df) > 0:

        gt_boxes = (
            gt_df[
                gt_df["window"]
                == window
            ]
            .reset_index(drop=True)
        )

    else:

        gt_boxes = pd.DataFrame()


    # Predictions
    pred_boxes = (
        pred_df[
            pred_df["window"]
            == window
        ]
        .reset_index(drop=True)
    )


    candidate_matches = []


    # Calculate ALL pairwise IoUs

    for gt_idx, gt in gt_boxes.iterrows():

        gt_box = [
            gt["x1"],
            gt["y1"],
            gt["x2"],
            gt["y2"]
        ]


        for pred_idx, pred in pred_boxes.iterrows():

            pred_box = [
                pred["x1"],
                pred["y1"],
                pred["x2"],
                pred["y2"]
            ]


            iou = calculate_iou(
                gt_box,
                pred_box
            )


            if iou >= IOU_THRESHOLD:

                candidate_matches.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx
                    )
                )


    # Greedy one-to-one matching
    # Highest IoU first

    candidate_matches.sort(
        key=lambda x: x[0],
        reverse=True
    )


    matched_gt = set()
    matched_pred = set()


    for (
        iou,
        gt_idx,
        pred_idx
    ) in candidate_matches:


        if (
            gt_idx not in matched_gt
            and
            pred_idx not in matched_pred
        ):

            matched_gt.add(
                gt_idx
            )

            matched_pred.add(
                pred_idx
            )


            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,

                    "result": "TP",

                    "iou": iou,

                    "gt_id": gt_idx,

                    "prediction_id": pred_idx,

                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


    # FALSE NEGATIVES
    # GT exists but model missed it

    for gt_idx in range(
        len(gt_boxes)
    ):

        if gt_idx not in matched_gt:

            evaluation_records.append(
                {
                    "window": window,

                    "result": "FN",

                    "iou": np.nan,

                    "gt_id": gt_idx,

                    "prediction_id":
                        np.nan,

                    "confidence":
                        np.nan
                }
            )


    # FALSE POSITIVES
    # Model predicted something not matching GT
    
    for pred_idx in range(
        len(pred_boxes)
    ):

        if pred_idx not in matched_pred:

            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,

                    "result": "FP",

                    "iou": np.nan,

                    "gt_id":
                        np.nan,

                    "prediction_id":
                        pred_idx,

                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


# 10. EVALUATION DATAFRAME

eval_df = pd.DataFrame(
    evaluation_records
)


TP = (
    eval_df["result"]
    == "TP"
).sum()

FP = (
    eval_df["result"]
    == "FP"
).sum()

FN = (
    eval_df["result"]
    == "FN"
).sum()


# 11. METRICS

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0
)


recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)


f1 = (
    2 * precision * recall
    /
    (precision + recall)

    if (
        precision + recall
    ) > 0

    else 0
)


matched_ious = (
    eval_df.loc[
        eval_df["result"]
        == "TP",
        "iou"
    ]
)


mean_iou = (
    matched_ious.mean()
    if len(matched_ious) > 0
    else np.nan
)


median_iou = (
    matched_ious.median()
    if len(matched_ious) > 0
    else np.nan
)

# 12. PRINT RESULTS

print("\n")
print("=" * 55)

print(
    "CHARACTER DETECTION EVALUATION"
)

print("=" * 55)

print(
    f"Images:               "
    f"{len(all_windows)}"
)

print(
    f"GT characters:        "
    f"{len(gt_df)}"
)

print(
    f"Predicted characters: "
    f"{len(pred_df)}"
)

print()

print(
    f"TP:                   {TP}"
)

print(
    f"FP:                   {FP}"
)

print(
    f"FN:                   {FN}"
)

print()

print(
    f"Precision:            "
    f"{precision:.3f}"
)

print(
    f"Recall:               "
    f"{recall:.3f}"
)

print(
    f"F1:                   "
    f"{f1:.3f}"
)

print(
    f"Mean matched IoU:     "
    f"{mean_iou:.3f}"
)

print(
    f"Median matched IoU:   "
    f"{median_iou:.3f}"
)

print("=" * 55)

# 13. SAVE DETAILED RESULTS

detail_path = (
    output_dir
    / "character_evaluation_details.csv"
)


eval_df.to_csv(
    detail_path,
    index=False
)


# 14. SAVE SUMMARY

summary_df = pd.DataFrame(
    [
        {
            "element": "character",

            "iou_threshold":
                IOU_THRESHOLD,

            "images":
                len(all_windows),

            "ground_truth":
                len(gt_df),

            "predictions":
                len(pred_df),

            "TP": TP,
            "FP": FP,
            "FN": FN,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "mean_matched_iou":
                mean_iou,

            "median_matched_iou":
                median_iou
        }
    ]
)


summary_path = (
    output_dir
    / "character_evaluation_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")

print(
    detail_path
)

print(
    summary_path
)

Images found: 82
Unique windows: 82

Prediction CSV columns:
['window', 'character_id', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2']

Predicted boxes: 140
Windows with predictions: 68

Prediction classes:
class
person    140
Name: count, dtype: int64

Label files: 82

GT character boxes: 173
Total windows for evaluation: 82


CHARACTER DETECTION EVALUATION
Images:               82
GT characters:        173
Predicted characters: 140

TP:                   104
FP:                   36
FN:                   69

Precision:            0.743
Recall:               0.601
F1:                   0.665
Mean matched IoU:     0.772
Median matched IoU:   0.775

Saved:
../02_data_results/evaluation_i_will_marry_your_brother_episode1/character_evaluation_details.csv
../02_data_results/evaluation_i_will_marry_your_brother_episode1/character_evaluation_summary.csv


### **2. Text regions**

In [7]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import re


# 1. Paths

labelstudio_dir = Path(
    "../02_data_results/labelstudio_i_will_marry_your_brother_episode1"
)

gt_dir = labelstudio_dir / "labels"

image_dir = Path(
    "../03_webtoon_raw/window_i_will_marry_your_brother_episode1"
)

prediction_csv = Path(
    "../02_data_results/text_i_will_marry_your_brother_episode1.csv"
)

output_dir = Path(
    "../02_data_results/evaluation_i_will_marry_your_brother_episode1"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# 2. Settings

# classes.txt:
# 0 = character
# 1 = panel
# 2 = text

TEXT_CLASS_ID = 2
IOU_THRESHOLD = 0.50


# 3. Extract window name

def get_window_name(filename):

    match = re.search(
        r"(window_\d+)",
        str(filename)
    )

    if match:
        return match.group(1)

    return Path(filename).stem


# 4. Calculate IoU

def calculate_iou(box1, box2):

    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])

    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if (
        x_right <= x_left
        or y_bottom <= y_top
    ):
        return 0.0

    intersection = (
        (x_right - x_left)
        * (y_bottom - y_top)
    )

    area1 = (
        (box1[2] - box1[0])
        * (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0])
        * (box2[3] - box2[1])
    )

    union = (
        area1
        + area2
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


# 5. Find all images

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
}

image_paths = [
    p
    for p in image_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

print("Images found:", len(image_paths))


image_map = {}

for path in image_paths:

    window = get_window_name(
        path.name
    )

    image_map[window] = path


print("Unique windows:", len(image_map))


# 6. Load model predictions

pred_df = pd.read_csv(
    prediction_csv
)

print("\nPrediction CSV columns:")
print(pred_df.columns.tolist())


possible_image_columns = [
    "image",
    "window",
    "filename",
    "file",
    "image_name",
    "file_name"
]

image_column = None

for col in possible_image_columns:

    if col in pred_df.columns:

        image_column = col
        break


if image_column is None:

    raise ValueError(
        "Could not find image/window column. "
        f"Available columns: {pred_df.columns.tolist()}"
    )


print(
    "Using image column:",
    image_column
)


pred_df["window_eval"] = (
    pred_df[image_column]
    .astype(str)
    .apply(get_window_name)
)


print(
    "\nPredicted boxes:",
    len(pred_df)
)

print(
    "Windows with predictions:",
    pred_df["window_eval"].nunique()
)


if "class" in pred_df.columns:

    print("\nPrediction classes:")

    print(
        pred_df["class"]
        .value_counts()
    )


# 7. Load Label Studio text ground truth

gt_records = []

label_files = sorted(
    gt_dir.glob("*.txt")
)

print(
    "\nLabel files:",
    len(label_files)
)


for label_file in label_files:

    window = get_window_name(
        label_file.name
    )

    if window not in image_map:

        print(
            "WARNING: image not found for",
            label_file.name
        )

        continue


    image_path = image_map[window]


    with Image.open(image_path) as img:

        W, H = img.size


    with open(
        label_file,
        "r"
    ) as f:

        lines = f.readlines()


    for line in lines:

        parts = (
            line
            .strip()
            .split()
        )


        if len(parts) != 5:
            continue


        class_id = int(
            parts[0]
        )


        if class_id != TEXT_CLASS_ID:
            continue


        x_center = float(
            parts[1]
        )

        y_center = float(
            parts[2]
        )

        width = float(
            parts[3]
        )

        height = float(
            parts[4]
        )


        x1 = (
            x_center
            - width / 2
        ) * W

        y1 = (
            y_center
            - height / 2
        ) * H

        x2 = (
            x_center
            + width / 2
        ) * W

        y2 = (
            y_center
            + height / 2
        ) * H


        gt_records.append(
            {
                "window": window,
                "image_width": W,
                "image_height": H,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            }
        )


gt_df = pd.DataFrame(
    gt_records
)


print(
    "\nGT text boxes:",
    len(gt_df)
)


# 8. Check windows

all_windows = sorted(
    image_map.keys()
)


print(
    "Total windows for evaluation:",
    len(all_windows)
)


# 9. Match ground truth and predictions

evaluation_records = []


for window in all_windows:

    if len(gt_df) > 0:

        gt_boxes = (
            gt_df[
                gt_df["window"]
                == window
            ]
            .reset_index(drop=True)
        )

    else:

        gt_boxes = pd.DataFrame()


    pred_boxes = (
        pred_df[
            pred_df["window_eval"]
            == window
        ]
        .reset_index(drop=True)
    )


    candidate_matches = []


    for gt_idx, gt in gt_boxes.iterrows():

        gt_box = [
            gt["x1"],
            gt["y1"],
            gt["x2"],
            gt["y2"]
        ]


        for pred_idx, pred in pred_boxes.iterrows():

            pred_box = [
                pred["x1"],
                pred["y1"],
                pred["x2"],
                pred["y2"]
            ]


            iou = calculate_iou(
                gt_box,
                pred_box
            )


            if iou >= IOU_THRESHOLD:

                candidate_matches.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx
                    )
                )


    candidate_matches.sort(
        key=lambda x: x[0],
        reverse=True
    )


    matched_gt = set()
    matched_pred = set()


    for (
        iou,
        gt_idx,
        pred_idx
    ) in candidate_matches:


        if (
            gt_idx not in matched_gt
            and
            pred_idx not in matched_pred
        ):

            matched_gt.add(
                gt_idx
            )

            matched_pred.add(
                pred_idx
            )


            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,
                    "result": "TP",
                    "iou": iou,
                    "gt_id": gt_idx,
                    "prediction_id": pred_idx,
                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


    for gt_idx in range(
        len(gt_boxes)
    ):

        if gt_idx not in matched_gt:

            evaluation_records.append(
                {
                    "window": window,
                    "result": "FN",
                    "iou": np.nan,
                    "gt_id": gt_idx,
                    "prediction_id": np.nan,
                    "confidence": np.nan
                }
            )


    for pred_idx in range(
        len(pred_boxes)
    ):

        if pred_idx not in matched_pred:

            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,
                    "result": "FP",
                    "iou": np.nan,
                    "gt_id": np.nan,
                    "prediction_id": pred_idx,
                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


# 10. Evaluation dataframe

eval_df = pd.DataFrame(
    evaluation_records
)


TP = (
    eval_df["result"]
    == "TP"
).sum()

FP = (
    eval_df["result"]
    == "FP"
).sum()

FN = (
    eval_df["result"]
    == "FN"
).sum()


# 11. Calculate metrics

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0
)

recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if (precision + recall) > 0
    else 0
)


matched_ious = (
    eval_df.loc[
        eval_df["result"] == "TP",
        "iou"
    ]
)


mean_iou = (
    matched_ious.mean()
    if len(matched_ious) > 0
    else np.nan
)

median_iou = (
    matched_ious.median()
    if len(matched_ious) > 0
    else np.nan
)


# 12. Print results

print("\n")
print("TEXT DETECTION EVALUATION")

print(
    f"Images:             {len(all_windows)}"
)

print(
    f"GT text boxes:      {len(gt_df)}"
)

print(
    f"Predicted boxes:    {len(pred_df)}"
)

print()

print(f"TP:                 {TP}")
print(f"FP:                 {FP}")
print(f"FN:                 {FN}")

print()

print(f"Precision:          {precision:.3f}")
print(f"Recall:             {recall:.3f}")
print(f"F1:                 {f1:.3f}")
print(f"Mean matched IoU:   {mean_iou:.3f}")
print(f"Median matched IoU: {median_iou:.3f}")


# 13. Save detailed results

detail_path = (
    output_dir
    / "text_evaluation_details.csv"
)

eval_df.to_csv(
    detail_path,
    index=False
)


# 14. Save summary

summary_df = pd.DataFrame(
    [
        {
            "element": "text",
            "iou_threshold": IOU_THRESHOLD,
            "images": len(all_windows),
            "ground_truth": len(gt_df),
            "predictions": len(pred_df),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "mean_matched_iou": mean_iou,
            "median_matched_iou": median_iou
        }
    ]
)


summary_path = (
    output_dir
    / "text_evaluation_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")
print(detail_path)
print(summary_path)

Images found: 82
Unique windows: 82

Prediction CSV columns:
['window', 'detection_id', 'label', 'confidence', 'x1', 'y1', 'x2', 'y2']
Using image column: window

Predicted boxes: 163
Windows with predictions: 69

Label files: 82

GT text boxes: 173
Total windows for evaluation: 82


TEXT DETECTION EVALUATION
Images:             82
GT text boxes:      173
Predicted boxes:    163

TP:                 148
FP:                 15
FN:                 25

Precision:          0.908
Recall:             0.855
F1:                 0.881
Mean matched IoU:   0.850
Median matched IoU: 0.867

Saved:
../02_data_results/evaluation_i_will_marry_your_brother_episode1/text_evaluation_details.csv
../02_data_results/evaluation_i_will_marry_your_brother_episode1/text_evaluation_summary.csv


### **3. Panel-like regions**

In [3]:
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import re


# 1. PATHS

# Label Studio export
labelstudio_dir = Path(
    "../02_data_results/labelstudio_i_will_marry_your_brother_episode1"
)

gt_dir = labelstudio_dir / "labels"

# IMPORTANT:
# use original webtoon windows for image dimensions
image_dir = Path(
    "../03_webtoon_raw/window_i_will_marry_your_brother_episode1"
)

# Panel model predictions
prediction_csv = Path(
    "../02_data_results/panel_i_will_marry_your_brother_episode1.csv"
)

# Evaluation output
output_dir = Path(
    "../02_data_results/evaluation_i_will_marry_your_brother_episode1"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# 2. SETTINGS

# classes.txt:
# 0 = character
# 1 = panel
# 2 = text

PANEL_CLASS_ID = 1

IOU_THRESHOLD = 0.50

# 3. FUNCTION: EXTRACT WINDOW NAME

def get_window_name(filename):
    """
    Examples:

    fcfc06f1-window_001.txt
        -> window_001

    window_001.png
        -> window_001
    """

    match = re.search(
        r"(window_\d+)",
        str(filename)
    )

    if match:
        return match.group(1)

    return Path(filename).stem


# 4. FUNCTION: CALCULATE IoU

def calculate_iou(box1, box2):
    """
    box format:
    [x1, y1, x2, y2]
    """

    x_left = max(
        box1[0],
        box2[0]
    )

    y_top = max(
        box1[1],
        box2[1]
    )

    x_right = min(
        box1[2],
        box2[2]
    )

    y_bottom = min(
        box1[3],
        box2[3]
    )

    # No overlap
    if (
        x_right <= x_left
        or y_bottom <= y_top
    ):
        return 0.0

    intersection = (
        (x_right - x_left)
        *
        (y_bottom - y_top)
    )

    area1 = (
        (box1[2] - box1[0])
        *
        (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0])
        *
        (box2[3] - box2[1])
    )

    union = (
        area1
        + area2
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


# 5. FIND ALL IMAGES

image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
}

image_paths = [
    p
    for p in image_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

print(
    "Images found:",
    len(image_paths)
)


# Create:
# window_001 -> image path

image_map = {}

for path in image_paths:

    window = get_window_name(
        path.name
    )

    image_map[window] = path


print(
    "Unique windows:",
    len(image_map)
)


# 6. LOAD MODEL PREDICTIONS

pred_df = pd.read_csv(
    prediction_csv
)

print("\nPrediction CSV columns:")
print(pred_df.columns.tolist())


# Convert image names:
# window_001.png
# ->
# window_001

pred_df["window"] = (
    pred_df["image"]
    .apply(get_window_name)
)


print(
    "\nPredicted boxes:",
    len(pred_df)
)

print(
    "Windows with predictions:",
    pred_df["window"].nunique()
)


# 7. LOAD LABEL STUDIO PANEL GT

gt_records = []

label_files = sorted(
    gt_dir.glob("*.txt")
)


print(
    "\nLabel files:",
    len(label_files)
)


for label_file in label_files:

    window = get_window_name(
        label_file.name
    )

    # Find corresponding image

    if window not in image_map:

        print(
            "WARNING: image not found for",
            label_file.name
        )

        continue


    image_path = image_map[window]

    # Read actual image dimensions
    with Image.open(image_path) as img:

        W, H = img.size


    # Read YOLO GT annotation

    with open(
        label_file,
        "r"
    ) as f:

        lines = f.readlines()


    for line in lines:

        parts = (
            line
            .strip()
            .split()
        )


        if len(parts) != 5:
            continue


        class_id = int(
            parts[0]
        )


        # PANEL ONLY
        if class_id != PANEL_CLASS_ID:
            continue


        x_center = float(
            parts[1]
        )

        y_center = float(
            parts[2]
        )

        width = float(
            parts[3]
        )

        height = float(
            parts[4]
        )


        # YOLO normalized xywh
        # ->
        # pixel xyxy

        x1 = (
            x_center
            - width / 2
        ) * W

        y1 = (
            y_center
            - height / 2
        ) * H

        x2 = (
            x_center
            + width / 2
        ) * W

        y2 = (
            y_center
            + height / 2
        ) * H


        gt_records.append(
            {
                "window": window,

                "image_width": W,
                "image_height": H,

                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            }
        )


gt_df = pd.DataFrame(
    gt_records
)


print(
    "\nGT panel boxes:",
    len(gt_df)
)


# 8. CHECK WINDOWS

all_windows = sorted(
    image_map.keys()
)


print(
    "Total windows for evaluation:",
    len(all_windows)
)

# 9. MATCH GT AND PREDICTIONS


evaluation_records = []


for window in all_windows:

    # --------------------------------------------
    # Ground truth
    # --------------------------------------------

    if len(gt_df) > 0:

        gt_boxes = (
            gt_df[
                gt_df["window"]
                == window
            ]
            .reset_index(drop=True)
        )

    else:

        gt_boxes = pd.DataFrame()


    # Predictions

    pred_boxes = (
        pred_df[
            pred_df["window"]
            == window
        ]
        .reset_index(drop=True)
    )


    candidate_matches = []


    # Calculate ALL pairwise IoUs

    for gt_idx, gt in gt_boxes.iterrows():

        gt_box = [
            gt["x1"],
            gt["y1"],
            gt["x2"],
            gt["y2"]
        ]


        for pred_idx, pred in pred_boxes.iterrows():

            pred_box = [
                pred["x1"],
                pred["y1"],
                pred["x2"],
                pred["y2"]
            ]


            iou = calculate_iou(
                gt_box,
                pred_box
            )


            if iou >= IOU_THRESHOLD:

                candidate_matches.append(
                    (
                        iou,
                        gt_idx,
                        pred_idx
                    )
                )


    # Greedy one-to-one matching
    # highest IoU first

    candidate_matches.sort(
        key=lambda x: x[0],
        reverse=True
    )


    matched_gt = set()
    matched_pred = set()


    for (
        iou,
        gt_idx,
        pred_idx
    ) in candidate_matches:


        if (
            gt_idx not in matched_gt
            and
            pred_idx not in matched_pred
        ):

            matched_gt.add(
                gt_idx
            )

            matched_pred.add(
                pred_idx
            )


            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,

                    "result": "TP",

                    "iou": iou,

                    "gt_id": gt_idx,

                    "prediction_id": pred_idx,

                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


    # FALSE NEGATIVES
    # GT exists but model missed it

    for gt_idx in range(
        len(gt_boxes)
    ):

        if gt_idx not in matched_gt:

            evaluation_records.append(
                {
                    "window": window,

                    "result": "FN",

                    "iou": np.nan,

                    "gt_id": gt_idx,

                    "prediction_id":
                        np.nan,

                    "confidence":
                        np.nan
                }
            )


    # FALSE POSITIVES
    # Model predicted something not matching GT

    for pred_idx in range(
        len(pred_boxes)
    ):

        if pred_idx not in matched_pred:

            pred_row = pred_boxes.iloc[
                pred_idx
            ]


            evaluation_records.append(
                {
                    "window": window,

                    "result": "FP",

                    "iou": np.nan,

                    "gt_id":
                        np.nan,

                    "prediction_id":
                        pred_idx,

                    "confidence":
                        pred_row.get(
                            "confidence",
                            np.nan
                        )
                }
            )


# 10. EVALUATION DATAFRAME

eval_df = pd.DataFrame(
    evaluation_records
)


TP = (
    eval_df["result"]
    == "TP"
).sum()

FP = (
    eval_df["result"]
    == "FP"
).sum()

FN = (
    eval_df["result"]
    == "FN"
).sum()


# 11. METRICS

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else 0
)


recall = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)


f1 = (
    2 * precision * recall
    /
    (precision + recall)

    if (
        precision + recall
    ) > 0

    else 0
)


matched_ious = (
    eval_df.loc[
        eval_df["result"]
        == "TP",
        "iou"
    ]
)


mean_iou = (
    matched_ious.mean()
    if len(matched_ious) > 0
    else np.nan
)


median_iou = (
    matched_ious.median()
    if len(matched_ious) > 0
    else np.nan
)


# 12. PRINT RESULTS

print("\n")
print("=" * 55)

print(
    "PANEL DETECTION EVALUATION"
)

print("=" * 55)

print(
    f"Images:            "
    f"{len(all_windows)}"
)

print(
    f"GT panels:         "
    f"{len(gt_df)}"
)

print(
    f"Predicted panels:  "
    f"{len(pred_df)}"
)

print()

print(
    f"TP:                {TP}"
)

print(
    f"FP:                {FP}"
)

print(
    f"FN:                {FN}"
)

print()

print(
    f"Precision:         "
    f"{precision:.3f}"
)

print(
    f"Recall:            "
    f"{recall:.3f}"
)

print(
    f"F1:                "
    f"{f1:.3f}"
)

print(
    f"Mean matched IoU:  "
    f"{mean_iou:.3f}"
)

print(
    f"Median matched IoU:"
    f"  {median_iou:.3f}"
)

print("=" * 55)


# 13. SAVE DETAILED RESULTS

detail_path = (
    output_dir
    / "panel_evaluation_details.csv"
)


eval_df.to_csv(
    detail_path,
    index=False
)


# 14. SAVE SUMMARY

summary_df = pd.DataFrame(
    [
        {
            "element": "panel",

            "iou_threshold":
                IOU_THRESHOLD,

            "images":
                len(all_windows),

            "ground_truth":
                len(gt_df),

            "predictions":
                len(pred_df),

            "TP": TP,
            "FP": FP,
            "FN": FN,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "mean_matched_iou":
                mean_iou,

            "median_matched_iou":
                median_iou
        }
    ]
)


summary_path = (
    output_dir
    / "panel_evaluation_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


print("\nSaved:")

print(
    detail_path
)

print(
    summary_path
)

Images found: 82
Unique windows: 82

Prediction CSV columns:
['image', 'panel_id', 'confidence', 'x1', 'y1', 'x2', 'y2']

Predicted boxes: 127
Windows with predictions: 69

Label files: 82

GT panel boxes: 236
Total windows for evaluation: 82


PANEL DETECTION EVALUATION
Images:            82
GT panels:         236
Predicted panels:  127

TP:                86
FP:                41
FN:                150

Precision:         0.677
Recall:            0.364
F1:                0.474
Mean matched IoU:  0.831
Median matched IoU:  0.873

Saved:
../02_data_results/evaluation_i_will_marry_your_brother_episode1/panel_evaluation_details.csv
../02_data_results/evaluation_i_will_marry_your_brother_episode1/panel_evaluation_summary.csv
